In [1]:

!pip install -U google-genai
!pip install requests
!pip install pandas
!pip install newspaper3k
!pip install lxml_html_clean
!pip install gtts
!pip install ipython

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
# ==========================================
# IMPORTING LIBRARIES
# ==========================================

from google import genai

import requests
import pandas as pd
import time

from newspaper import Article

from gtts import gTTS

from IPython.display import Audio

In [3]:
# ==========================================
# GEMINI API CONFIGURATION
# ==========================================

GEMINI_API_KEY = "******************************"

client = genai.Client(api_key=GEMINI_API_KEY)

In [4]:
# ==========================================
# NEWS API CONFIGURATION
# ==========================================

NEWS_API_KEY = "**********************"

In [5]:
# ==========================================
# FETCHING LATEST NEWS
# ==========================================

url = f"https://newsapi.org/v2/top-headlines?country=us&apiKey={NEWS_API_KEY}"

response = requests.get(url)

news_data = response.json()

articles = news_data["articles"]

print("LATEST NEWS HEADLINES")
print("=" * 50)

for i, article in enumerate(articles[:5]):

    print(f"\nNews {i+1}")

    print("Title:", article["title"])

    print("Description:", article["description"])

LATEST NEWS HEADLINES

News 1
Title: SpaceX Starship Flight 12 live launch updates: 1st Starship V3 launch scrubbed at last minute - Space
Description: SpaceX is now targeting a Friday, May 22, launch for its newest Starship design, the Starship V3 megarocket after a launch scrub. See our latest updates here.

News 2
Title: Stephen Colbert’s Final ‘Late Show’ Guests Include Paul McCartney - Deadline
Description: The studio audience was "moved" by the farewell for Colbert, who also had Ryan Reynolds and Paul Rudd among the guests for his final show.

News 3
Title: Trump says he’ll ‘try’ to attend son’s wedding this weekend but it’s ‘not good timing’ - NBC News
Description: The president said that due to the Iran war and “other things,” he wasn’t sure if he’d be attending the nuptials of Donald Trump Jr. and Bettina Anderson this weekend.

News 4
Title: Anker’s new earbuds have the best call quality I’ve ever heard - The Verge
Description: Soundcore earbuds have outperformed their price 

In [6]:
# ==========================================
# AI SUMMARIZATION FUNCTION
# ==========================================

def summarize_news(text):

    prompt = f"""
    Summarize the following news article in 3 concise bullet points.

    News:
    {text}
    """

    retries = 3

    for attempt in range(retries):

        try:

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            return response.text

        except Exception as e:

            print(f"Attempt {attempt+1} failed:", e)

            time.sleep(15)

    return "Summary not available due to API limits."

In [7]:
# ==========================================
# GENERATING AI SUMMARIES
# ==========================================

print("\nAI GENERATED SUMMARIES")
print("=" * 50)

for i, article in enumerate(articles[:3]):

    content = str(article["title"]) + " " + str(article["description"])

    summary = summarize_news(content)

    print("\n-----------------------------------")

    print(f"News {i+1}")

    print("TITLE:", article["title"])

    print("\nSUMMARY:")

    print(summary)

    time.sleep(10)


AI GENERATED SUMMARIES

-----------------------------------
News 1
TITLE: SpaceX Starship Flight 12 live launch updates: 1st Starship V3 launch scrubbed at last minute - Space

SUMMARY:
Here's a summary of the article in 3 concise bullet points:

*   The inaugural launch attempt for SpaceX's Starship V3 megarocket was scrubbed at the last minute.
*   SpaceX is now targeting Friday, May 22, for the next launch attempt.
*   This flight marks the debut of the newest Starship V3 design.

-----------------------------------
News 2
TITLE: Stephen Colbert’s Final ‘Late Show’ Guests Include Paul McCartney - Deadline

SUMMARY:
Here's a summary of the article in 3 concise bullet points:

*   Paul McCartney was a key guest on Stephen Colbert's final 'Late Show'.
*   Ryan Reynolds and Paul Rudd also appeared on the farewell episode.
*   The studio audience was reportedly "moved" by Colbert's send-off.

-----------------------------------
News 3
TITLE: Trump says he’ll ‘try’ to attend son’s weddin

In [8]:
# ==========================================
# STORING RESULTS IN DATAFRAME
# ==========================================

news_list = []

for article in articles[:3]:

    title = article["title"]

    content = str(article["title"]) + " " + str(article["description"])

    summary = summarize_news(content)

    news_list.append({
        "Title": title,
        "Summary": summary
    })
    
    print(f"Processed: {title}")

    time.sleep(10)

df = pd.DataFrame(news_list)

df

Processed: SpaceX Starship Flight 12 live launch updates: 1st Starship V3 launch scrubbed at last minute - Space
Processed: Stephen Colbert’s Final ‘Late Show’ Guests Include Paul McCartney - Deadline
Processed: Trump says he’ll ‘try’ to attend son’s wedding this weekend but it’s ‘not good timing’ - NBC News


,Title,Summary
0,SpaceX Starship Flight 12 live launch updates:...,Here's a summary of the news article in 3 conc...
1,Stephen Colbert’s Final ‘Late Show’ Guests Inc...,Here's the summary in 3 concise bullet points:...
2,Trump says he’ll ‘try’ to attend son’s wedding...,Here's the summary in 3 concise bullet points:...


In [9]:
# ==========================================
# SAVING SUMMARIZED NEWS
# ==========================================
import os

os.makedirs("data", exist_ok=True)
df.to_csv("data/summarized_news.csv", index=False)

print("CSV File Saved Successfully!")

CSV File Saved Successfully!


In [10]:
# ==========================================
# ARTICLE EXTRACTION FUNCTION
# ==========================================

def extract_article(url):

    try:

        article = Article(url)

        article.download()

        article.parse()

        return article.text

    except Exception as e:

        print("Error extracting article:", e)

        return None

In [11]:
# ==========================================
# TEXT TO SPEECH FUNCTION
# ==========================================

def generate_voice(summary):

    os.makedirs("audio", exist_ok=True)
    
    tts = gTTS(text=summary, lang='en')

    tts.save("audio/news_summary.mp3")

    return Audio("audio/news_summary.mp3", autoplay=True)

In [12]:
# ==========================================
# URL BASED AI NEWS SUMMARIZER
# ==========================================

news_url = input("Enter News Article URL: ")

article_text = extract_article(news_url)

if article_text:

    print("\nARTICLE EXTRACTED SUCCESSFULLY\n")

    summary = summarize_news(article_text[:5000])

    print("\nAI GENERATED SUMMARY\n")

    print(summary)

else:

    print("Failed to extract article.")

Enter News Article URL:  https://www.channelnewsasia.com/today/ground-up/rice-farming-indonesia-agriculture-methane-emissions-6133921



ARTICLE EXTRACTED SUCCESSFULLY

Attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

AI GENERATED SUMMARY

Here's a summary in 3 concise bullet points:

*   Indonesian rice farmer Mr. Kasno and 170 others are trialling a new farming method developed by Singapore's Temasek Life Sciences Laboratory (TLL).
*   The new method involves a different fertilizer, a more weather-resilient rice variety, and Alternate Wetting and Drying (AWD) irrigation.
*   This approach has significantly increased rice yields (from 6-7 to 8-9 tonnes/hectare) and reduced water/fertilizer use for farmers, while also aiming to cut methane emissions.


In [13]:
# ==========================================
# GENERATING VOICE SUMMARY
# ==========================================

generate_voice(summary)